# Lakeflow Connect + DLT-Meta Demo

This demo shows how to:
1. Create Lakeflow Connect (LFC) pipelines that produce **streaming tables** in a Unity Catalog schema
2. Configure DLT-Meta to use those LFC streaming tables as the **source for bronze tables**

**Reference:** [lfcddemo-one-click-notebooks/lfc/db/lfcdemo-database.ipynb](https://github.com/rsleedbx/lfcddemo-one-click-notebooks/blob/cleanup/lfc/db/lfcdemo-database.ipynb) for creating the LFC gateway and ingestion pipelines.

## Step 1: Create LFC Pipelines (Reference Implementation)

Run the [lfcdemo-database.ipynb](https://github.com/rsleedbx/lfcddemo-one-click-notebooks/blob/cleanup/lfc/db/lfcdemo-database.ipynb) notebook to create:

- **Gateway pipeline** (CDC mode) – captures changes from source database
- **Ingestion pipeline** – creates streaming tables in `{target_catalog}.{target_schema}`

Example output schema: `main.lfcdemo_staging` with streaming tables `intpk`, `dtix`.

## Step 2: DLT-Meta Onboarding – Bronze from LFC Streaming Tables

Once LFC pipelines are running, configure DLT-Meta to read from the **streaming tables** as delta sources.

In [ ]:
# DLT-Meta onboarding config: bronze source = LFC streaming table
# Replace placeholders with your catalog, schema, and table names

onboarding_lfc = {
    "data_flow_id": "300",
    "data_flow_group": "A1",
    "source_format": "delta",  # LFC streaming tables are Delta
    "source_details": {
        "source_table": "intpk",
        "source_path_dev": "main.lfcdemo_staging.intpk",  # catalog.schema.table
    },
    "bronze_catalog_dev": "dev_catalog",
    "bronze_database_dev": "lfc_bronze",
    "bronze_table": "intpk_from_lfc",
    "bronze_table_path_dev": "/Volumes/dev_catalog/dltmeta/data/bronze/intpk_from_lfc",
    "bronze_reader_options": {
        "format": "delta"
    },
    "bronze_database_quarantine_dev": "dev_catalog.lfc_bronze",
    "bronze_quarantine_table": "intpk_quarantine",
    "silver_catalog_dev": "dev_catalog",
    "silver_database_dev": "lfc_silver",
    "silver_table": "intpk_clean",
}

print("Example onboarding entry for LFC streaming table as bronze source:")
import json
print(json.dumps(onboarding_lfc, indent=2))

## Step 3: Run DLT-Meta Onboard

Save the config to an onboarding JSON file and run:

```bash
databricks labs dlt-meta onboard --onboarding_file_path <path> --uc_catalog_name dev_catalog ...
```

## Flow Summary

```
Source DB (SQL Server/PostgreSQL/MySQL)
    |
    v
LFC Gateway + Ingestion Pipelines
    |
    v
Streaming tables: {catalog}.{schema}.intpk, dtix, ...
    |
    v  source_format: delta, source_path_dev: catalog.schema.table
DLT-Meta Bronze Tables
    |
    v
DLT-Meta Silver Tables
```